# FML 2026 Challenge — Analysis and Prediction over Bike Sharing Data

## 0. Setup

In [ ]:
import os

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, adjusted_rand_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.utils import resample
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

sns.set_theme(style="whitegrid")

RANDOM_STATE = 42
DATA_DIR = os.environ.get("DATA_DIR", "../csv")

def rmse(y, p):
    return mean_squared_error(y, p) ** 0.5

## 1. Data Understanding and Exploration

### 1.1 Loading datasets

In [ ]:
bike = pd.read_csv(f"{DATA_DIR}/bikes.csv")
types = pd.read_csv(f"{DATA_DIR}/bike_types.csv")
city = pd.read_csv(f"{DATA_DIR}/cities.csv")
station = pd.read_csv(f"{DATA_DIR}/stations.csv")
station_status = pd.read_csv(f"{DATA_DIR}/stations_status.csv")
trips = pd.read_csv(f"{DATA_DIR}/trips_small.csv")

The project contains six tables:
- bike: individual bicycles
- types: technical bike-type metadata
- city: bike-sharing schemes and their geographic metadata
- station: station metadata and capacity
- station_status: repeated station-state snapshots over time
- trips: individual journeys

### 1.2 Inventory of each table

In [ ]:
def data_invent(data):
    """Shape, duplicates, dtypes, missing values, unique values, descriptive statistics."""
    
    inventory = pd.DataFrame({
        "Data type": data.dtypes,
        "Missing values": data.isna().sum(),
        "Missing value in %": data.isna().mean() * 100,
        "Unique values": data.nunique(),
    })
    
    print(f"Shape of the dataframe is : {data.shape}\n")
    print(f"The duplicate rows are : {data.duplicated().sum()}\n")
    display(inventory)

    print("\nThe descriptive statistics :")
    display(data.describe(include="all").T)

#### Bikes dataframe

In [ ]:
data_invent(bike)

* The dataset contains exactly 88,436 unique id values and zero duplicate rows. 
* There are exactly 121 unique bike_type_id values. The dataframe has 0% missing values (NaNs) across all three columns. 
* While there are 88,436 bikes, there are only 72,278 unique computer_ids. The descriptive statistics show the minimum computer_id is 0.0. 

#### Bike types dataframe

In [ ]:
data_invent(types)

* The dataframe contains 121 records with exactly 121 unique id values and zero duplicate rows. We can use it as a primary key for joining with the bikes table.

* Exactly 30 rows (24.8%) are simultaneously missing fundamental attributes: vehicle_image, name, form_factor, rider_capacity, and propulsion_type. This can refer to unclassified or deleted bike configurations in the system. We will need to decide whether to impute these as "Unknown" or exclude them later.

* The high missing rates for battery_capacity (81.8%) and max_range (87.6%) are logical, as human-powered bikes do not possess these metrics. The descriptive statistics shows that the 15 non-null records for max_range all share the exact same value (60000.0). 

* There is a strong many-to-one relationship in the types. While there are 121 technical ids, there are only 46 unique commercial names and 3 form_factors. This suggests that many technical IDs simply represent similar hardware of the same fundamental bike model.

* The descriptive statistics highlight that bikes are designed for either 1 or 2 riders, and the electric bikes utilize just four standard battery sizes ranging tightly between 500 Wh and 690 Wh.

In [ ]:
for col in ["propulsion_type", "form_factor"]:
    print(col, "->", types[col].value_counts(dropna=False).to_dict())

#### City dataframe

In [ ]:
data_invent(city)

* The dataframe contains 267 records with 0% missing values across all columns and no duplicate rows. Both id and name are primary identifiers for the bike-sharing schemes.

*  While there are 267 distinct scheme names, there are only 264 unique lat values and 266 unique lon values. This slight discrepancy confirms that some bike-sharing schemes share the exact same approximate city center coordinates. 

* The return_to_official_only variable is a  boolean feature indicating whether bikes must be left at a station. This distinguishes between strictly station-based and free-floating setups.

* The network spans 15 distinct countries and 9 time zones. The bounding coordinates—from a latitude of 27.8° to 57.7° and a longitude of -15.5° to 33.4°—confirm a broad footprint encompassing diverse European urban environments.

#### Trips dataframe

In [ ]:
data_invent(trips)

* The dataset contains 1,161,376 total trip records. The core spatial coordinates (lon_start, lat_start, lon_end, lat_end), temporal data (time_start), and routing metrics (duration, distance) contain zero missing values, providing a highly reliable foundation for geospatial mapping and time-series aggregation.

* station_id_start is missing for 89,215 records and station_id_end is missing for 121,904 records. A NULL station ID indicates a free-floating start or end. They tell us whether a user utilized an official dock or left the bicycle in an ad-hoc location.

* The battery_start and battery_end variables are missing roughly 85% of the dataset (over 992,000 records). This high sparsity is due to the electric bicycles. For the electric subset, the median battery drain per trip is approximately 6%.

* The descriptive statistics reveal a median trip duration of 720 seconds (12 minutes) and a median distance of roughly 1,346 meters (1.35 km). However, the maximum duration extends to 28,740 seconds (nearly 8 hours) and the maximum distance to roughly 20 kilometers. 

#### Station dataframe

In [ ]:
data_invent(station)

* The id column contains exactly 2,932 unique values for the 2,932 rows, functioning as a perfect primary key. However, there are only 2,855 unique station names, indicating that approximately 77 stations share names. 

* The rack_locks column is entirely empty (NaN), meaning it provides zero predictive value and should be dropped during preprocessing. Furthermore, the terminal_type (indicating hardware generation) is missing for 587 stations (~20%). 

* The descriptive statistics reveal a large portion of stations (1,220 out of 2,932) have exactly 0 bike_racks. This implies that these are not physical docks, but rather virtual stations or designated geofenced parking zones for free-floating setups. 

* The presence of special_racks (charging docks) shows a heavy positive skew, with the vast majority of stations having 0 (up through the 75th percentile), but a maximum of 12. 

#### Station status dataframe

In [ ]:
data_invent(station_status)

* With nearly 2.93 million snapshot records and absolutely zero null values across all eight columns.

* The time column is currently stored as a float64 UNIX timestamp. To make this useful it must be converted into Pandas datetime objects to find cyclical features such as hour-of-day, day-of-week, and weekday-vs-weekend.

*  The descriptive statistics show massive variance in station scale. While the median station has 4 free racks and 5 bikes available, the maximums stretch to 70 free racks and 189 available bikes. 

* The maintenance boolean flag indicates if a station is offline. 

### 1.3 Relationships between tables

In [ ]:
primary_key_checks = pd.DataFrame({
    "table": ["bike", "types", "city", "station"],
    "key": ["id", "id", "id", "id"],
    "rows": [len(bike), len(types), len(city), len(station)],
    "unique_key_values": [
        bike["id"].nunique(), 
        types["id"].nunique(),
        city["id"].nunique(), 
        station["id"].nunique()
    ],
})
primary_key_checks["unique_key"] = primary_key_checks["rows"] == primary_key_checks["unique_key_values"]
display(primary_key_checks)

In [ ]:
def fk_check(child, child_col, parent, parent_col, label, allow_null=False):
    values = child[child_col]
    if allow_null:
        values = values.dropna()
    matched = values.isin(parent[parent_col])
    return {
        "relationship": label, 
        "checked_values": len(values), 
        "matched": int(matched.sum()),
        "unmatched": int((~matched).sum()),
        "match_percent": round(matched.mean() * 100, 3) if len(values) else np.nan
    }

fk_results = pd.DataFrame([
    fk_check(bike, "bike_type_id", types, "id", "bikes.bike_type_id -> bike_types.id"),
    fk_check(station, "city_id", city, "id", "stations.city_id -> cities.id"),
    fk_check(station_status, "station_id", station, "id", "station_status.station_id -> stations.id"),
    fk_check(trips, "bike_id", bike, "id", "trips.bike_id -> bikes.id"),
    fk_check(trips, "city_id", city, "id", "trips.city_id -> cities.id"),
    fk_check(trips, "station_id_start", station, "id", "trips.station_id_start -> stations.id", allow_null=True),
    fk_check(trips, "station_id_end", station, "id", "trips.station_id_end -> stations.id", allow_null=True),
])
display(fk_results)

In [ ]:
stations_without_status = set(station["id"]) - set(station_status["station_id"])
status_station_ids_not_in_metadata = set(station_status["station_id"]) - set(station["id"])

print("Stations in metadata:", station["id"].nunique())
print("Stations represented in status:", station_status["station_id"].nunique())
print("Stations with no status observations:", len(stations_without_status))
print("Status station IDs absent from station metadata:", len(status_station_ids_not_in_metadata))

### 1.4 Bikes and bike types

In [ ]:
print("Number of individual bikes:", bike["id"].nunique())
print("Number of unique bike types:", types["id"].nunique())

##### Distribution of bikes per type

In [ ]:
bike_profile = bike.merge(
    types[["id", "name", "propulsion_type", "form_factor", "rider_capacity", "battery_capacity", "max_range"]],
    left_on="bike_type_id", 
    right_on="id", 
    how="left", 
    validate="many_to_one",
    indicator=True, 
    suffixes=("_bike", "_type")
)
display(bike_profile["_merge"].value_counts())

In [ ]:
bike_distribution = (
    bike_profile.groupby(["name", "propulsion_type"], dropna=False)
    .agg(
        individual_bikes=("id_bike", "count"), 
        technical_type_ids=("bike_type_id", "nunique")
    )
    .sort_values("individual_bikes", ascending=False)
)
display(bike_distribution.head(20))

In [ ]:
plt.figure(figsize=(11, 6))
bike_distribution.head(15)["individual_bikes"].sort_values().plot(kind="barh")
plt.title("15 most represented bike types in the fleet")
plt.xlabel("Number of individual bikes")
plt.tight_layout()
plt.show()

##### Bike type characteristics

In [ ]:
display(types.groupby("propulsion_type", dropna=False).agg(
    type_records=("id", "count"),
    unique_names=("name", "nunique"),
    non_missing_battery=("battery_capacity", "count"),
    non_missing_range=("max_range", "count"),
    battery_values=("battery_capacity", lambda x: sorted(x.dropna().unique().tolist())),
    range_values=("max_range", lambda x: sorted(x.dropna().unique().tolist()))
))

In [ ]:
types["form_factor"].value_counts(dropna=False).sort_index()

In [ ]:
types["rider_capacity"].value_counts(dropna=False).sort_index()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
types["propulsion_type"].value_counts(dropna=False).plot(kind="bar", ax=axes[0], title="Bike types by propulsion")
types["form_factor"].value_counts(dropna=False).plot(kind="bar", ax=axes[1], title="Bike types by form factor")
types["rider_capacity"].value_counts(dropna=False).sort_index().plot(kind="bar", ax=axes[2], title="Rider capacity")

axes[0].set_xlabel("Propulsion type")
axes[1].set_xlabel("Form factor")
axes[2].set_xlabel("Rider capacity")

for ax in axes:
    ax.set_ylabel("Number of technical types")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Identifies the most reused names across different technical IDs.
name_reuse = (
    types.groupby("name", dropna=False)
    .agg(
        technical_ids=("id", "nunique"),
        propulsion_types=("propulsion_type", lambda x: x.nunique(dropna=False)),
        form_factors=("form_factor", lambda x: x.nunique(dropna=False))
    )
    .sort_values("technical_ids", ascending=False)
)
display(name_reuse.head(20))

* The 88,436 bikes span 121 types, but distribution is highly concentrated: 47% are simply named `with Framelock`. Many names describe hardware features rather than actual bike types

* 30 technical types (about 25%) lack specifications, covering 1,050 bikes (plus 253 named Unknown). We keep them as they are referenced in trip data

* Commercial names are often shared across multiple technical IDs (10 for `Rower z fotelikiem dziecięcym`), and identical bike types appear under various spellings and languages, so the raw name column requires cleaning before being used as a category

* As expected, battery capacity and range are only recorded for electric models

### 1.5 Cities

In [ ]:
country_counts = city["country"].value_counts()
display(country_counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
country_counts.plot(kind="bar", ax=axes[0], title="Number of bike-sharing schemes by country", xlabel="Country", ylabel="Number of schemes")
sns.scatterplot(data=city, x="lon", y="lat", hue="country", legend=False, s=35, ax=axes[1])

axes[1].set_title("Geographic distribution of scheme reference coordinates")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.show()

In [ ]:
print("Unique schemes:", city["id"].nunique())
print("Unique countries:", city["country"].nunique())
print("Unique time zones:", city["timezone"].nunique())
print("Coordinate pairs shared by several schemes:", int((city.groupby(["lat", "lon"]).size() > 1).sum()))

In [ ]:
display(city[(city["lat"] == 51.2469) & (city["lon"] == 22.56970)])

Both records have the same approximate city-centre coordinates, but they represent separate scheme records. This is not necessarily a duplicate or an error. It could be two schemes operating in the same geographical area

In [ ]:
rto_counts = city["return_to_official_only"].value_counts().sort_index()
rto_counts.plot(
    kind="pie",
    autopct=lambda p: f"{p:.1f}%\n({int(round(p * rto_counts.sum() / 100))})",
    startangle=90,
    ylabel="",
)
plt.title("Schemes by official-return requirement")
plt.tight_layout()
plt.show()

* In total, there are 267 schemes in 15 different countries, and 9 different timezone. Germany is the country that has the more schemes (91).

* 200 out of 267 schemes (about 75%) require bikes to be returned to an official station, the other 67 allow free-floating returns. 

### 1.6 Stations and their infrastructure

In [ ]:
print("Number of stations:", station["id"].nunique())

In [ ]:
display(pd.DataFrame(station["name"].value_counts()))
display(pd.DataFrame(station["terminal_type"].value_counts()))
display(pd.DataFrame(station["place_type"].value_counts()))

In [ ]:
display(pd.DataFrame(station.groupby('terminal_type')['bike_racks'].count()))
display(pd.DataFrame(station.groupby(['bike_racks','terminal_type']).size()))

In [ ]:
repeated_station_names = (
    station[station["name"].duplicated(keep=False)]
    .sort_values(["name", "city_id", "id"])
)

display(repeated_station_names[["id", "city_id", "name", "lon", "lat"]].head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

station["terminal_type"].value_counts(dropna=False).plot(
    kind="bar", 
    ax=axes[0], 
    title="Terminal type",
    xlabel="Terminal type"
)

station["place_type"].value_counts(dropna=False).sort_index().plot(
    kind="bar", 
    ax=axes[1], 
    title="Place type codes",
    xlabel="Place type"
)

for ax in axes:
    ax.set_ylabel("Number of stations")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
display(pd.crosstab(station["bike_racks"], station["terminal_type"]))

In [ ]:
station["total_racks_raw"] = station["bike_racks"] + station["special_racks"]
plt.figure(figsize=(12, 5))
sns.boxplot(data=station, x=station["place_type"].astype(str), y="total_racks_raw")
plt.title("Distribution of total racks by place type code")
plt.xlabel("Place type")
plt.show()

top_stations = station.sort_values("total_racks_raw", ascending=False).head(10)
plt.figure(figsize=(10, 5))
sns.barplot(data=top_stations, x="total_racks_raw", y="name")
plt.title("Top 10 stations by total capacity")
plt.show()

##### Regular and Special Rack Capacity

In [ ]:
station[['bike_racks','special_racks']]

In [ ]:
station['bike_racks'].value_counts()

In [ ]:
print("Stations with 0 regular racks:", int((station["bike_racks"] == 0).sum()), "/", len(station))
print("Stations with negative special_racks:", int((station["special_racks"] < 0).sum()))
print("Missing coordinates:", int(station[["lon", "lat"]].isna().any(axis=1).sum()))

In [ ]:
# Negative special_racks values
display(station[station["special_racks"] < 0])

In [ ]:
# Zero Rack Stations
display(station[station['bike_racks'] == 0])
print("Terminal types for stations with 0 regular racks:", station[station['bike_racks'] == 0]['terminal_type'].value_counts())

In [ ]:
# Comparing zero rack and non zero rack stations
station_status_with_capacity = station_status.merge(
    station[["id", "bike_racks", "special_racks", "terminal_type", "place_type"]],
    left_on="station_id",
    right_on="id",
    how="left",
    validate="many_to_one"
)

# New column to indicate stations with zero regular racks
station_status_with_capacity["zero_regular_racks"] = (
    station_status_with_capacity["bike_racks"] == 0
)

zero_rack_status_comparison = station_status_with_capacity.groupby("zero_regular_racks").agg(
    total_observations=("station_id", "size"),
    unique_stations_count=("station_id", "nunique"),
    avg_total_bikes_present=("bikes", "mean"),
    avg_rentable_bikes=("bikes_available_to_rent", "mean"),
    percent_time_completely_empty=("bikes_available_to_rent", lambda x: (x == 0).mean() * 100),
    avg_free_regular_racks=("free_racks", "mean"),
    avg_free_special_racks=("free_special_racks", "mean")
)

In [ ]:
display(zero_rack_status_comparison)

#### Station infrastructure by scheme (merge with cities)

In [ ]:
station_city = station.merge(
    city, 
    left_on="city_id", 
    right_on="id", 
    how="left",
    suffixes=("_station", "_city"), 
    validate="many_to_one", 
    indicator=True
)

print(station_city["_merge"].value_counts())

In [ ]:
# Identifying stations with unmatched city IDs
unmatched_station_city = station_city.loc[
    station_city["_merge"] != "both",
    ["id_station", "city_id", "name_station"]
]
display(unmatched_station_city)

In [ ]:
# Number of stations per scheme
stations_per_city = (
    station_city.groupby(['city_id', 'name_city'])
    .agg(number_of_stations=('id_station', 'count'))
    .sort_values('number_of_stations', ascending=False)
)

display(stations_per_city)

plt.figure(figsize=(10, 6))
stations_per_city.head(15)["number_of_stations"].sort_values().plot(kind="barh")
plt.title("15 schemes with the most stations")
plt.xlabel("Number of stations")
plt.ylabel("Scheme")
plt.tight_layout()
plt.show()

In [ ]:
# Capacity per scheme
capacity_per_city = (
    station_city.groupby(["city_id", "name_city"])
    .agg(
        number_of_stations=("id_station", "nunique"),
        total_regular_racks=("bike_racks", "sum"),
        total_special_racks_raw=("special_racks", "sum"),
        mean_regular_racks=("bike_racks", "mean"),
        median_regular_racks=("bike_racks", "median"),
        stations_with_negative_special_racks=("special_racks", lambda x: (x < 0).sum())
    )
    .sort_values("total_regular_racks", ascending=False)
)
display(capacity_per_city.head(20))

In [ ]:
# Average station capacity per scheme
station_city.groupby(['city_id', 'name_city']).agg(
    average_regular_racks=('bike_racks', 'mean'),
    median_regular_racks=('bike_racks', 'median'),
    average_special_racks=('special_racks', 'mean')
).sort_values('average_regular_racks', ascending=False)

In [ ]:
# Groups stations by city to compute the proportion of zero-rack stations
# Sorted by highest percentage

zero_rack_analysis = (
    station_city.assign(
        zero_rack=station_city['bike_racks'].eq(0)
    )
    .groupby(['city_id', 'name_city'])
    .agg(
        total_stations=('id_station', 'count'),
        zero_rack_stations=('zero_rack', 'sum')
    )
)

zero_rack_analysis['zero_rack_percentage'] = (
    100 * zero_rack_analysis['zero_rack_stations']
    / zero_rack_analysis['total_stations']
)

zero_rack_analysis.sort_values('zero_rack_percentage', ascending=False)

* 1,220 of 2,932 stations (41.6%) have zero regular racks. They are probably virtual free-floating zones (as confirmed by their terminal types which are mostly `free` or `sign`)

* The proportion of these stations varies significantly across cities: some networks have zero physical racks, while others are entirely dock-based.

* 32 stations have a negative `special_racks`, all of which feature a 7-inch terminal type

### 1.7 Station status: consistency

In [ ]:
maintenance_analysis = station_status.groupby('maintenance').agg(
    observations=('station_id', 'size'),
    mean_bikes=('bikes', 'mean'),
    mean_booked=('booked_bikes', 'mean'),
    mean_available=('bikes_available_to_rent', 'mean'),
    zero_available=(
        'bikes_available_to_rent',
        lambda x: (x == 0).mean() * 100
    )
)

maintenance_analysis

In [ ]:
raw_difference = station_status["bikes"] - station_status["booked_bikes"]
clamped_expected = raw_difference.clip(lower=0)

raw_match = station_status["bikes_available_to_rent"].eq(raw_difference)
clamped_match = station_status["bikes_available_to_rent"].eq(clamped_expected)

formula_comparison = pd.DataFrame({
    "comparison": ["bikes - booked_bikes", "max(bikes - booked_bikes, 0)"],
    "matching_rows": [int(raw_match.sum()), int(clamped_match.sum())],
    "non_matching_rows": [int((~raw_match).sum()), int((~clamped_match).sum())],
    "match_percent": [raw_match.mean() * 100, clamped_match.mean() * 100]
})
display(formula_comparison)

In [ ]:
remaining_formula_cases = station_status.loc[
    ~clamped_match,
    ["station_id", "time", "bikes", "booked_bikes",
     "bikes_available_to_rent", "maintenance",
     "free_racks", "free_special_racks"]
]
display(remaining_formula_cases)

In [ ]:
negative_bookings = station_status[
    station_status['booked_bikes'] < 0
]

print("Rows with negative booked_bikes:", len(negative_bookings))
negative_bookings.groupby('maintenance').size()

In [ ]:
display(negative_bookings["booked_bikes"].value_counts().sort_index())

In [ ]:
status_capacity = station_status.merge(
    station[['id', 'bike_racks', 'special_racks']],
    left_on='station_id',
    right_on='id',
    how='left',
    validate='many_to_one'
)

status_capacity['declared_capacity_raw'] = (
    status_capacity['bike_racks']
    + status_capacity['special_racks']
)

status_capacity['observed_occupied_and_free'] = (
    status_capacity['bikes']
    + status_capacity['free_racks']
    + status_capacity['free_special_racks']
)

status_capacity["capacity_difference_raw"] = (
    status_capacity["declared_capacity_raw"] -
    status_capacity["observed_occupied_and_free"]
)

status_capacity['capacity_difference_raw'].describe()

In [ ]:
print("Unmatched station metadata rows:", int(status_capacity["id"].isna().sum()))
print("Stations with negative special_racks:", int((station["special_racks"] < 0).sum()))
print("Status rows with negative special-rack metadata:",
      int((status_capacity["special_racks"] < 0).sum()))
display(status_capacity["capacity_difference_raw"].describe())
display(status_capacity["capacity_difference_raw"].value_counts().head(20))
del status_capacity

##### Correlation between station status measurements

In [ ]:
status_numeric_columns = [
    "bikes", "booked_bikes", "bikes_available_to_rent",
    "free_racks", "free_special_racks"
]
status_correlations = station_status[status_numeric_columns].corr()
display(status_correlations)

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(status_correlations, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation between station-status numeric variables")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
sns.boxplot(data=station_status, x="maintenance",
            y="bikes_available_to_rent", showfliers=False)
plt.title("Available bikes by maintenance status")
plt.xlabel("Maintenance")
plt.ylabel("Bikes available to rent")
plt.tight_layout()
plt.show()

### 1.8 Station status: temporal structure

In [ ]:
station_status['datetime'] = pd.to_datetime(
    station_status['time'],
    unit='s',
    utc=True
)

print("Period:", station_status['datetime'].min(), "to", station_status['datetime'].max())
print("Time span:", station_status['datetime'].max() - station_status['datetime'].min())
print("Unique stations in status:", station_status['station_id'].nunique())
print("Duplicate station-time rows:", station_status.duplicated(["station_id", "time"]).sum())

In [ ]:
status = station_status.sort_values(['station_id', 'time']).copy()
status['gap_min'] = status.groupby('station_id')['time'].diff()/60
status['gap_min'].describe()

In [ ]:
station_frequency = status.groupby('station_id').agg(
    observations=('time', 'count'),
    median_interval=('gap_min', 'median'),
    mean_interval=('gap_min', 'mean')
)

station_frequency.describe().T

In [ ]:
print("Number of duplicates:",status.duplicated(subset=['station_id', 'time']).sum())

In [ ]:
intervals = status["gap_min"].dropna()

station_status["month"] = station_status["datetime"].dt.to_period("M").astype(str)
station_status["hour_utc"] = station_status["datetime"].dt.hour
station_status["weekday_utc"] = station_status["datetime"].dt.day_name()

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
axes = axes.flatten()

axes[0].hist(intervals[intervals <= 24 * 60], bins=60)
axes[0].set_title("Within-station observation intervals (up to 24 h)")
axes[0].set_xlabel("Minutes between consecutive observations")
axes[0].set_ylabel("Number of intervals")

axes[1].hist(station_frequency["observations"], bins=50)
axes[1].set_title("Number of status observations per station")
axes[1].set_xlabel("Observations per station")
axes[1].set_ylabel("Number of stations")

station_status.groupby("month").size().plot(
    kind="bar", ax=axes[2], title="Snapshots by month (UTC)"
)
axes[2].set_xlabel("Month")
axes[2].set_ylabel("Number of snapshots")
axes[2].tick_params(axis="x", rotation=45)

station_status.groupby("hour_utc").size().plot(
    kind="bar", ax=axes[3], title="Snapshots by hour of day (UTC)"
)
axes[3].set_xlabel("Hour of day (UTC)")
axes[3].set_ylabel("Number of snapshots")

plt.tight_layout()
plt.show()

### 1.9 Trips

In [ ]:
trip_validity = pd.Series({
    "duration_nonpositive": (trips["duration"] <= 0).sum(),
    "distance_nonpositive": (trips["distance"] <= 0).sum(),
    "invalid_start_longitude": (~trips["lon_start"].between(-180, 180)).sum(),
    "invalid_end_longitude": (~trips["lon_end"].between(-180, 180)).sum(),
    "invalid_start_latitude": (~trips["lat_start"].between(-90, 90)).sum(),
    "invalid_end_latitude": (~trips["lat_end"].between(-90, 90)).sum()
}, name="count")
display(trip_validity)

In [ ]:
display(trips[["duration", "distance"]].corr())

In [ ]:
trip_summary = trips[["duration", "distance"]].describe(
    percentiles=[.5, .75, .9, .95, .99]
).T
display(trip_summary)

In [ ]:
sample = trips.sample(n=min(50000, len(trips)), random_state=RANDOM_STATE)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
axes[0].scatter(sample["distance"] / 1000, sample["duration"] / 60, alpha=0.15, s=8)
axes[0].set_title("Duration vs distance")
axes[0].set_xlabel("km")
axes[0].set_ylabel("min")

axes[1].hist(trips["duration"] / 60, bins=60)
axes[1].set_title("Trip duration (min)")
axes[1].set_xlabel("Minutes")

axes[2].hist(trips["distance"] / 1000, bins=60)
axes[2].set_title("Trip distance (km)")
axes[2].set_xlabel("Kilometers")

plt.tight_layout()
plt.show()

In [ ]:
trip_profile = (
    trips.merge(
        bike[["id", "bike_type_id"]].rename(columns={"id": "bike_id"}), 
        on="bike_id",
        how="left", 
        validate="many_to_one"
    ).merge(
        types[["id", "propulsion_type"]].rename(columns={"id": "bike_type_id"}), 
        on="bike_type_id", 
        how="left", 
        validate="many_to_one"
    )
)

battery_missing_by_propulsion = trip_profile.groupby("propulsion_type", dropna=False).agg(
    trips=("bike_id", "size"),
    missing_battery_start=("battery_start", lambda x: x.isna().mean() * 100),
    missing_battery_end=("battery_end", lambda x: x.isna().mean() * 100),
    median_battery_start=("battery_start", "median"),
    median_battery_end=("battery_end", "median")
)
display(battery_missing_by_propulsion)

In [ ]:
trip_profile["start_station_missing"] = trip_profile["station_id_start"].isna()
trip_profile["end_station_missing"] = trip_profile["station_id_end"].isna()

In [ ]:
free_floating_by_propulsion = trip_profile.groupby(
    "propulsion_type", dropna=False
).agg(
    trips=("bike_id", "size"),
    missing_start_station_percent=("start_station_missing", "mean"),
    missing_end_station_percent=("end_station_missing", "mean")
) * 1
free_floating_by_propulsion[[
    "missing_start_station_percent", "missing_end_station_percent"
]] *= 100
display(free_floating_by_propulsion)

In [ ]:
trip_city = trips.merge(
    city[["id", "name", "country"]],
    left_on="city_id",
    right_on="id",
    how="left",
    validate="many_to_one",
    suffixes=("_trip", "_scheme")
)

trip_city["start_station_missing"] = trip_city["station_id_start"].isna()
trip_city["end_station_missing"] = trip_city["station_id_end"].isna()

free_floating_by_scheme = trip_city.groupby(
    ["city_id", "name", "country"], dropna=False
).agg(
    trips=("time_start", "size"),
    missing_start_station_percent=("start_station_missing", "mean"),
    missing_end_station_percent=("end_station_missing", "mean")
)

free_floating_by_scheme[[
    "missing_start_station_percent", "missing_end_station_percent"
]] *= 100
display(free_floating_by_scheme.sort_values("trips", ascending=False).head(20))

## 2. Data Preparation

### 2.1 Bike types

In [ ]:
types_c = types.copy()
types_c["propulsion_type"] = types_c["propulsion_type"].replace({"electric_assist": "electric"})
types_c["form_factor"] = types_c["form_factor"].replace({"bicycle": "regular", "cargo_bicycle": "cargo"})

# electric types with missing battery/range: mean of electric types with the same form factor
battery_cols = ["max_range", "battery_capacity"]
is_electric = types_c["propulsion_type"] == "electric"
for form in ["regular", "cargo"]:
    m = is_electric & (types_c["form_factor"] == form)
    types_c.loc[m, battery_cols] = types_c.loc[m, battery_cols].fillna(types_c.loc[m, battery_cols].mean())

# remaining gaps: unknown types (kept: referenced by bikes) and human-powered bikes
string_cols = ["vehicle_image", "name", "description", "form_factor", "propulsion_type"]
numeric_cols = ["rider_capacity", "max_range", "battery_capacity"]
types_c[string_cols] = types_c[string_cols].fillna("Unknown")
types_c[numeric_cols] = types_c[numeric_cols].fillna(0)

print("Missing values left:", int(types_c.isna().sum().sum()))
print(types_c["propulsion_type"].value_counts().to_dict(), types_c["form_factor"].value_counts().to_dict())

#### Cleaning the bike-type names

The 46 commercial names represent bike types under various spellings and languages. They are normalized using prioritized keyword rules (most specific first). Names describing only locks or hardware generations (for example `with Framelock`) are classified as standard city bikes. 

In [ ]:
def clean_bike_type_name(name):
    
    if pd.isna(name) or str(name).strip().lower() in {"", "unknown", "nan"}:
        return "Unknown"
    
    n = re.sub(r"\s+", " ", str(name)).strip().lower()
    rules = [
        ("E-cargo bike",         r"^e-?cargo"),
        ("Cargo bike",           r"cargo"),
        ("Tandem",               r"tandem"),
        ("Handbike",             r"handbike"),
        ("Tricycle",             r"tr[oó]jko[lł]owy|tricycle"),
        ("Bike with child seat", r"fotelik|child|sjedalica"),
        ("Kids bike",            r"kinder|rowerek"),
        ("E-bike",               r"^e-?bike|^e-?smartbike|^e-?classicbike|elektri"),
        ("City bike",            r"smartbike|classic|comfort|tourenfahrrad|usedomrad|wa[łl]em|kardan|framelock|fork lock"),
    ]

    for label, pattern in rules:
        if re.search(pattern, n):
            return label
    return "Other"

In [ ]:
types_c["bike_type_group"] = types_c["name"].map(clean_bike_type_name)
print("Names mapped to 'Other' (to be reviewed):", sorted(types_c.loc[types_c["bike_type_group"] == "Other", "name"].unique()))
types_c["bike_type_group"].value_counts(dropna=False).sort_index()

In [ ]:
# Name with bike_type_group mapping
mapping = (types_c.groupby(["bike_type_group", "name"])
           .agg(technical_ids=("id", "count"), bikes=("id", lambda s: int(bike["bike_type_id"].isin(s).sum())))
           .sort_values(["bike_type_group", "bikes"], ascending=[True, False]))
display(mapping)

In [ ]:
# Summary with bike_type_group
display(types_c.groupby("bike_type_group").agg(
    raw_names=("name", "nunique"), technical_ids=("id", "count"),
    bikes=("id", lambda s: int(bike["bike_type_id"].isin(s).sum()))).sort_values("bikes", ascending=False))

In [ ]:
# Checking the relationship between bike_type_group and propulsion_type
display(pd.crosstab(types_c["bike_type_group"], types_c["propulsion_type"]))

* The raw names collapse into 10 groups: *City bike*, *E-bike*, *Cargo bike*, *E-cargo bike*, *Bike with child seat*, *Kids bike*, *Tandem*, *Tricycle*, *Handbike* and *Unknown*. No name falls into *Other*, so the rules cover every name present in the data 

* The cross-table with `propulsion_type` is a consistency check: the *E-bike* and *E-cargo bike* groups should be electric and the *City bike* group human-powered. 

### 2.2 Stations

In [ ]:
def prepare_station(station_raw):
    station_c = station_raw.drop(columns=["rack_locks", "total_racks_raw"], errors="ignore").copy()

    station_c["negative_special_racks_flag"] = (station_c["special_racks"] < 0).astype(int)
    station_c["zero_regular_racks_flag"] = (station_c["bike_racks"] == 0).astype(int)

    station_c["terminal_type"] = station_c["terminal_type"].fillna("Unknown")

    return station_c

In [ ]:
station_c = prepare_station(station)

station_c = station_c.merge(
    city[["id", "name", "country", "timezone", "return_to_official_only"]].rename(
        columns={"id": "city_id", "name": "scheme"}),
    on="city_id", 
    how="left", 
    validate="many_to_one"
)

station_c["return_to_official_only"] = station_c["return_to_official_only"].astype(int)

print("Shape:", station_c.shape)
print("Missing values:", int(station_c[["terminal_type", "country", "timezone"]].isna().sum().sum()))

### 2.3 Station status

In [ ]:
STATION_INFO = station_c.set_index("id")

# Cleans raw station status data, adds UTC time features (hour, weekend), 
# and computes occupancy metrics like fill rate, empty, and full flags.
def prepare_status(raw, station_info):
    
    s = raw.drop_duplicates(["station_id", "time"]).copy()

    s = s[s["station_id"].isin(station_info.index)].reset_index(drop=True)
    
    s["datetime"] = pd.to_datetime(s["time"], unit="s", utc=True).astype("datetime64[ns, UTC]")
    s["hour_utc"] = s["datetime"].dt.hour
    s["day_of_week_utc"] = s["datetime"].dt.dayofweek
    
    s["maintenance"] = s["maintenance"].astype(int)
    
    return s

status = prepare_status(station_status, STATION_INFO)
print(status.shape)
print("Period (UTC):", status["datetime"].min(), "->", status["datetime"].max())
status.head()

### 2.4 Trips

In [ ]:
def clean_trips(trips, bike, types_c, city):
    t = trips.copy()

    # Types
    t["station_id_start"] = t["station_id_start"].astype("Int64")
    t["station_id_end"] = t["station_id_end"].astype("Int64")

    t["time_start_dt"] = pd.to_datetime(t["time_start"], unit="s", utc=True)
    t["duration_min"] = t["duration"] / 60
    t["distance_km"] = t["distance"] / 1000

    t["free_start"] = t["station_id_start"].isna().astype(int)
    t["free_end"] = t["station_id_end"].isna().astype(int)

    t["hour_utc"] = t["time_start_dt"].dt.hour
    t["day_of_week_utc"] = t["time_start_dt"].dt.dayofweek
    t["time_end_dt"] = t["time_start_dt"] + pd.to_timedelta(t["duration_min"], unit="min")

    # Merges trips with bike, types, and city 
    t = (
        t.merge(
            bike[["id", "bike_type_id"]].rename(columns={"id": "bike_id"}),
            on="bike_id", how="left", validate="many_to_one",
        )
        .merge(
            types_c[["id", "name", "bike_type_group", "propulsion_type",
                     "form_factor", "rider_capacity", "max_range", "battery_capacity"]]
            .rename(columns={"id": "bike_type_id", "name": "bike_type_name"}),
            on="bike_type_id", how="left", validate="many_to_one",
        )
        .merge(
            city[["id", "name", "country", "timezone"]]
            .rename(columns={"id": "city_id", "name": "bike_sharing_system"}),
            on="city_id", how="left", validate="many_to_one",
        )
    )

    mask = t["propulsion_type"].isin(["electric", "human"]) & t["form_factor"].isin(["regular", "cargo"])
    t = t[mask].reset_index(drop=True)
    trips_c = t.drop(columns=["time_start", "duration", "distance"])

    return trips_c

trips_c = clean_trips(trips, bike, types_c, city)
trips_c.head()

### 2.5 Exploration of the prepared data

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
trips_c.groupby("country").size().sort_values().plot(kind="barh", ax=axes[0, 0], title="Number of trips per country")
sns.lineplot(data=trips_c, x="hour_utc", y="duration_min", hue="propulsion_type", estimator="mean", errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Mean trip duration by UTC hour of the day")
sns.boxplot(data=trips_c, x="propulsion_type", y="distance_km", showfliers=False, ax=axes[1, 0])
axes[1, 0].set_title("Distance by propulsion type")
trips_c["propulsion_type"].value_counts().plot(kind="pie", autopct="%1.1f%%", ax=axes[1, 1], title="Trips by propulsion type")
axes[1, 1].set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
rto = (trips_c.merge(city[["id", "return_to_official_only"]], left_on="city_id", right_on="id", how="left")
       .groupby("return_to_official_only")
       .agg(trips=("bike_id", "size"), free_start_pct=("free_start", lambda x: x.mean() * 100),
            free_end_pct=("free_end", lambda x: x.mean() * 100)))
display(rto)

In [ ]:
type_summary = (trips_c.groupby("bike_type_group")
                .agg(trips=("bike_id", "size"), bikes=("bike_id", "nunique"),
                     median_duration_min=("duration_min", "median"), median_distance_km=("distance_km", "median"),
                     electric_share=("propulsion_type", lambda s: (s == "electric").mean()))
                .sort_values("trips", ascending=False))
display(type_summary.round(2))

**Insights**

* Germany and Poland together account for more than half of the trips, and 85% of the trips are on human-powered bikes (15% electric)

* Electric trips are longer in distance and in time.

* In schemes that allow free returns, 10.6% of the trips start and 15.3% end outside a station. Even in schemes that require official returns, 7.1% start and 9.5% end outside a station. The flag `return_to_official_only` therefore indicates a rule, not a guarantee.

## 3. Unsupervised tasks: Bike Trip and Station Profiles

### 3.1 Part A - Clustering Bike Trips

#### Study 1 - Station profiles in the two largest cities

For each of the two cities with the most stations, every station is described by its coordinates (`lat`, `lon`), its activity (`volume_total` = departures + arrivals) and its attractiveness (`ratio_attractivite` = arrivals / departures). After standardisation, a KMeans with 2 clusters groups the stations with similar location and usage.

In [ ]:
city_names = city.set_index('id')['name']
city_names

In [ ]:
city_stats = (trips_c.groupby('city_id')
              .agg(nb_stations=('station_id_start', 'nunique'),
                   nb_trips=('bike_id', 'size'),
                   share_electric=('propulsion_type', lambda s: (s == 'electric').mean()))
              .nlargest(2, 'nb_stations'))
city_stats.rename(index=city_names).round(3)

We find out that these two cities we found either have only electric bikes for Barcelona or non electric bikes Lublin.

In [ ]:
features_a = ['lat', 'lon', 'volume_total', 'ratio_attractivite']
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
results_a = []

for ax, cid in zip(axes, city_stats.index):
    city_trips = trips_c[trips_c['city_id'] == cid]

    st = city_trips.groupby('station_id_start').agg(
        nb_departs=('bike_id', 'count'), lat=('lat_start', 'median'), lon=('lon_start', 'median'))
    st['nb_arrivals'] = city_trips.groupby('station_id_end').size()
    st = st.dropna()
    st['volume_total'] = st['nb_departs'] + st['nb_arrivals']
    st['ratio_attractivite'] = st['nb_arrivals'] / st['nb_departs']

    X_city = StandardScaler().fit_transform(st[features_a])
    st['cluster'] = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=10).fit_predict(X_city)
    st['city'] = city_names[cid]
    results_a.append(st)

    sns.scatterplot(data=st, x='lon', y='lat', hue='cluster', size='volume_total',
                    sizes=(20, 500), ax=ax)
    ax.set_title(f"{city_names[cid]} (silhouette = {silhouette_score(X_city, st['cluster']):.2f})")
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)

plt.tight_layout()
plt.show()

pd.concat(results_a).groupby(['city', 'cluster']).agg(
    n_stations=('lat', 'size'), lat=('lat', 'mean'), lon=('lon', 'mean'),
    mean_volume=('volume_total', 'mean'), mean_ratio=('ratio_attractivite', 'mean')).round(2)

In [ ]:
ks = [2, 3, 4, 5]
scores_a = {}
fig, axes = plt.subplots(len(results_a), len(ks), figsize=(5 * len(ks), 5 * len(results_a)))

for row, st in zip(axes, results_a):
    city_label = st['city'].iloc[0]
    X_city = StandardScaler().fit_transform(st[features_a])
    for ax, k in zip(row, ks):
        labels = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(X_city)
        scores_a.setdefault(city_label, {})[k] = silhouette_score(X_city, labels)
        sns.scatterplot(x=st['lon'], y=st['lat'], hue=labels, size=st['volume_total'],
                        sizes=(20, 300), palette='Set1', alpha=0.7, ax=ax, legend=False)
        ax.set_title(f"{city_label} - k={k} (silhouette = {scores_a[city_label][k]:.2f})")

plt.tight_layout()
plt.show()

pd.DataFrame(scores_a).rename_axis('k').round(3)

#### Choice of k and interpretation (k = 2)

**Choice of k.** We tested k = 2 to 5 using the silhouette score, and also looked at the maps. In **Barcelona**, the score is high and stable from k=2 to k=4 (0.52 to 0.55), then drops at k=5 (0.45): the two-zone structure is clear and k=3 or 4 adds almost nothing. In **Lublin**, the score stays low whatever k is (0.23 to 0.33): the stations form a continuum from the center to the periphery, and a larger k only cuts it into slices. We therefore keep **k = 2** for both cities: it is the simplest choice, consistent across the two cities, and interpretable.

**The special case k = 2.** With k=2, KMeans simply splits the stations in two along their direction of greatest variation: it cannot reveal more than two profiles. In Barcelona, the split corresponds to two geographically separate zones, which explains the high score. In Lublin, it only separates a very active zone from a calmer one, without a clear boundary: the low score (0.23) shows that this is a convenient partition, not two truly distinct types of stations.

**Interpretation of the results (k = 2)**

- **Barcelona**: the north-east group has a high arrivals/departures ratio (7.3 on average). It receives far more bikes than it sends out, which looks like a destination zone, i.e. a work or activity area. Departures come mostly from a single hub (about 49% of all departures), which acts as a point of origin. I cannot say that this hub is a residential area: it could also be a depot or a distribution point.

- **Lublin**: the ratio is close to 1 in both groups (0.96 and 1.17). The stations send out as many bikes as they receive, so we can speak neither of a work zone nor of a residential zone. Both groups are mixed zones. The central group is simply more active (mean volume of 481 versus 314).

### 3.2 Part B - Clustering Bike-Sharing Stations

The goal is to detect whether there are residential stations, office stations, employment hubs, highly active stations, etc.

#### Feature engineering on station status

In [ ]:
status_fe = status.copy()

# Weekend flag (Monday=0, Sunday=6), from the UTC day of week computed in section 2.3
status_fe['weekend'] = status_fe['day_of_week_utc'] >= 5

In [ ]:
# 'capacity' = total number of places in a station
status_fe['capacity'] = status_fe['bikes'] + status_fe['free_racks']
n_before = len(status_fe)
status_fe = status_fe[status_fe['capacity'] > 0]
print("Snapshots removed because the station has no capacity:", n_before - len(status_fe))

In [ ]:
# fill = rate of bikes in a station
status_fe['fill'] = status_fe['bikes'] / status_fe['capacity']

In [ ]:
# empty = 1 if no bike is available to rent, 0 else
status_fe['empty'] = (status_fe['bikes_available_to_rent'] == 0).astype(int)
# full = 1 if the station is full, 0 else
status_fe['full'] = (status_fe['free_racks'] == 0).astype(int)

In [ ]:
by_station = status_fe.groupby('station_id')

station_feat = pd.DataFrame({
    'fill_mean': by_station['fill'].mean(),
    'fill_std': by_station['fill'].std(),
    'pct_empty': by_station['empty'].mean(),
    'pct_full': by_station['full'].mean(),
    'pct_maint': by_station['maintenance'].mean(),
})
# fill_std: a high value indicates a station whose stock changes a lot, a low value a stable one

In [ ]:
# time slots of the day (UTC hours)
status_fe['slot'] = pd.cut(status_fe['hour_utc'], [-1, 5, 9, 15, 19, 23],
                           labels=['nuit', 'matin', 'midi', 'soir', 'late'])
status_fe

In [ ]:
slots = status_fe.pivot_table(index='station_id', columns='slot', values='fill', aggfunc='mean', observed=False)
slots

In [ ]:
station_feat = station_feat.join(slots.add_prefix('fill_'))
station_feat

In [ ]:
# Week vs weekend: mean fill rate of each group
weekend_fill = status_fe.groupby(['station_id', 'weekend'])['fill'].mean().unstack()
weekend_fill

In [ ]:
# if diff_we > 0, the station is more in demand during the weekend than during the rest of the week
# if diff_we < 0, the station is more in demand on weekdays
# else, similar usage all week long
station_feat['diff_we'] = weekend_fill[True] - weekend_fill[False]
station_feat

In [ ]:
station_feat = station_feat.join(STATION_INFO[['bike_racks', 'lon', 'lat']])
#reduce the mean and std using log(x+1)
#log1p can handle null values. And, the distribution  of "bike_racks" is asymetric: alot of meadum-sized station and few very large ones.
# so we use log1p to bring the distribution closer to a more symmetrical form. 
station_feat['bike_racks'] = np.log1p(station_feat['bike_racks'])

#### Choice of k

In [ ]:
# lon and lat are not used for the clustering (only to draw the clusters)
feat_cols = [c for c in station_feat.columns if c not in ['lon', 'lat']]
print(feat_cols)
X_stations = StandardScaler().fit_transform(station_feat[feat_cols].dropna())

for k in range(2, 9):
    # n_init=10: the algorithm restarts 10 times with different centers
    km = KMeans(k, n_init=10, random_state=RANDOM_STATE).fit(X_stations)
    # km.inertia_: sum of the squared distances between each station and the center of its cluster
    print(k, silhouette_score(X_stations, km.labels_), km.inertia_)

#### Final clustering (k = 4)

In [ ]:
station_feat_ok = station_feat[feat_cols].dropna().copy()
km_stations = KMeans(4, n_init=10, random_state=RANDOM_STATE).fit(X_stations)
station_feat_ok['cluster'] = km_stations.labels_
# Add lon and lat to draw the clusters
station_feat_ok = station_feat_ok.join(station_feat[['lon', 'lat']])
print(station_feat_ok['cluster'].value_counts().sort_index())

In [ ]:
# Description of each cluster
cluster_profile = station_feat_ok.groupby('cluster')[feat_cols].mean().round(2)
print(cluster_profile.T)

- **Cluster 0**: fill_mean = 0.34, active cluster, bike_racks = 2.39 (nearly 10 docks)
- **Cluster 1**: fill_mean = 1, always full, bike_racks = 0.05 (capacity too low)
- **Cluster 2**: fill_mean = 0.62, active cluster, pct_empty = 0.02 (rarely empty)
- **Cluster 3**: pct_maint = 0.99, out-of-service cluster

In [ ]:
# Standardized version
cluster_profile_z = (pd.DataFrame(X_stations, columns=feat_cols, index=station_feat_ok.index)
                     .groupby(station_feat_ok['cluster']).mean())
sns.heatmap(cluster_profile_z.T, annot=True, cmap='RdBu_r', center=0)
plt.title("Cluster profiles")
plt.show()

In [ ]:
status_fe_cl = status_fe.merge(station_feat_ok[['cluster']], left_on='station_id', right_index=True)
hourly_fill = status_fe_cl.groupby(['cluster', 'hour_utc'])['fill'].mean().reset_index()
sns.lineplot(data=hourly_fill, x='hour_utc', y='fill', hue='cluster', palette="Set1", marker='o')
plt.title("Fill rate per hour (UTC) and per cluster")
plt.show()

In [ ]:
sns.scatterplot(data=station_feat_ok, x='lon', y='lat', hue='cluster', palette='Set1', s=40, alpha=0.8)
plt.title("Geographic representation of clusters")
plt.show()

#### Stability check: KMeans vs Ward

In [ ]:
ward = AgglomerativeClustering(4).fit_predict(X_stations)
print("ARI KMeans vs ward :", adjusted_rand_score(km_stations.labels_, ward))
# ARI close to 1 means both methods find the same clustering ==> stable clustering

## 4. Supervised Task: Predict Station Availability

### 4.1 Preparation of the Dataset

In [ ]:
def preprocessing_status(status_raw, station_raw):

    # Prepare station_status
    station = prepare_station(station_raw)
    station_info = station.set_index("id")
    s = prepare_status(status_raw, station_info)
    s = s.sort_values("datetime").reset_index(drop=True)
    
    # Get the last record of the station
    s["end"] = s.groupby("station_id")["datetime"].transform("max")

    # Creation of the target variable 
    s["target_date"] = s["datetime"] + pd.Timedelta(hours=1)

    # Future state of the station (the next row) / The target
    future_state = s[["station_id", "datetime", "bikes_available_to_rent"]].rename(
                       columns={"datetime": "found_date", "bikes_available_to_rent": "y"})
    s = pd.merge_asof(s, future_state, left_on="target_date", right_on="found_date",
                      by="station_id", direction="backward")

    # Next state after the current state to avoid long data gaps
    next_state = s[["station_id", "datetime"]].rename(columns={"datetime": "next_date"})
    s = pd.merge_asof(s, next_state, left_on="target_date", right_on="next_date",
                      by="station_id", direction="forward")

    s = s.dropna(subset=["y"])

    # Remove the rows where the gap is too big 
    # Ensure it is not a a breakdown
    s = s[s["next_date"].notna() & (s["next_date"] - s["found_date"] <= pd.Timedelta("24h"))]
    
    # Remove the last row of each station (no records after)
    s = s[s["datetime"] + pd.Timedelta(hours=1) <= s["end"]]
    s = s[~s["free_special_racks"] < 0]

    return s.sort_values(["station_id", "datetime"]).reset_index(drop=True), station_info

def add_features(s, station_info):
    s = s.copy()

    # Historique : observation précédente de la même station
    by_station = s.groupby("station_id")
    s["previous_available_bikes"] = by_station["bikes_available_to_rent"].shift(1)
    s["previous_booked_bikes"] = by_station["booked_bikes"].shift(1)
    s["minutes_since_previous_observation"] = by_station["time"].diff() / 60
    s = s.dropna(subset=["previous_available_bikes"])

    # Temps
    s["month_utc"] = s["datetime"].dt.month
    s["hour_sin"] = np.sin(2 * np.pi * s["hour_utc"] / 24)
    s["hour_cos"] = np.cos(2 * np.pi * s["hour_utc"] / 24)

    # Station (STATION_INFO vient de la section 2.2 : terminal_type déjà rempli, flag négatif déjà calculé)
    s = s.join(station_info[["terminal_type", "place_type", "bike_racks", "special_racks",
                             "negative_special_racks_flag", "lon", "lat"]], on="station_id")
    s["special_racks"] = s["special_racks"].clip(lower=0)
    s["total_declared_racks"] = s["bike_racks"] + s["special_racks"]

    return pd.get_dummies(s, columns=["terminal_type"], drop_first=True, dtype=int).reset_index(drop=True)

In [ ]:
data, s = preprocessing_status(station_status, station)
data = add_features(data, s)
data.info()

In [ ]:
BASE_FEATURES = ["bikes", "booked_bikes", "free_racks", "free_special_racks", "maintenance", "day_of_week_utc", "hour_utc", "bikes_available_to_rent"]

NEW_FEATURES = ["previous_available_bikes", "previous_booked_bikes", "minutes_since_previous_observation",
                "month_utc", "hour_sin", "hour_cos",
                "place_type", "bike_racks", "special_racks", "total_declared_racks",
                "negative_special_racks_flag", "lon", "lat"]

TERMINAL_FEATURES = [c for c in data.columns if c.startswith("terminal_type_")]

FEATURES = BASE_FEATURES + NEW_FEATURES + TERMINAL_FEATURES
TARGET = "y"

first_df = data[FEATURES + [TARGET]]
first_df.head()

### 4.2 Model Training

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(first_df[FEATURES], first_df[TARGET], test_size=0.3, shuffle=False, random_state=RANDOM_STATE)
model_evaluation = {}

x_train_sample, y_train_sample = resample(x_train, y_train, n_samples=2000, random_state=RANDOM_STATE)

In [ ]:
def optimize_params(model, param_grid, x_train, y_train, x_test, y_test, scoring='r2', cv=3, n_iter=10, random_state=RANDOM_STATE):
    
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        n_iter=n_iter,
        scoring=scoring,
        cv=cv,
        n_jobs=-1,
        random_state=random_state
    )
    random_search.fit(x_train, y_train)
    best_params = random_search.best_params_
    
    return best_params

def evaluate_model(model, x_train, y_train, x_test, y_test):
    model.fit(x_train, y_train)
    y_pred = model.predict(x_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return mae, rmse, r2

#### KNN

In [ ]:
params = {
    'n_neighbors': [3, 5, 7, 9, 11, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_model = KNeighborsRegressor()
best_params = optimize_params(knn_model, params, 
                              x_train_sample, y_train_sample, x_test, y_test)

knn_model = KNeighborsRegressor(**best_params)
mae, rmse, r2 = evaluate_model(knn_model, x_train, y_train, x_test, y_test)

model_evaluation["KNN"] = {
    "best_params": best_params,
    "mae": mae,
    "rmse": rmse,
    "r2": r2
}

#### Decision Tree

In [ ]:
params = {
    "max_depth": [None, 5, 10, 15],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4]
}

decision_tree_model = DecisionTreeRegressor()
best_params = optimize_params(decision_tree_model, params, 
                              x_train_sample, y_train_sample, x_test, y_test)

decision_tree_model = DecisionTreeRegressor(**best_params)
mae, rmse, r2 = evaluate_model(decision_tree_model, x_train, y_train, x_test, y_test)

model_evaluation["Decision Tree"] = {
    "best_params": best_params,
    "mae": mae,
    "rmse": rmse,
    "r2": r2
}

#### Random Forest

In [ ]:
params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 5, 10],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

random_forest_model = RandomForestRegressor()
best_params = optimize_params(random_forest_model, params, 
                              x_train_sample, y_train_sample, x_test, y_test)
                              
random_forest_model = RandomForestRegressor(**best_params)
mae, rmse, r2 = evaluate_model(random_forest_model, x_train, y_train, x_test, y_test)

model_evaluation["Random Forest"] = {
    "best_params": best_params,
    "mae": mae,
    "rmse": rmse,
    "r2": r2
}

#### Gradient Boosting

In [ ]:
params = {
    "n_estimators": [100, 200, 300],
    "learning_rate": [0.1, 0.01, 0.001],
    "max_depth": [3, 5, 7],
    "num_leaves": [7, 15, 31],      
    "min_child_samples": [10, 20, 30] 
}

gradient_boosting_model = LGBMRegressor()
best_params = optimize_params(gradient_boosting_model, params,
                              x_train_sample, y_train_sample, x_test, y_test)

gradient_boosting_model = LGBMRegressor(**best_params)
mae, rmse, r2 = evaluate_model(gradient_boosting_model, x_train, y_train, x_test, y_test)

model_evaluation["LightGradientBoosting"] = {
    "best_params": best_params,
    "mae": mae,
    "rmse": rmse,
    "r2": r2
}

#### XGBoost

In [ ]:
params_xgb = {
    'max_depth': list((range(3,12))),
    'alpha': [0,0.001, 0.01,0.1,1],
    'subsample': [0.5,0.75,1],
    'learning_rate': np.linspace(0.01,0.5, 10),
    'n_estimators': [10, 25, 40]
    }
    
xgb_model = XGBRegressor()
best_params = optimize_params(xgb_model, params_xgb,
                              x_train_sample, y_train_sample, x_test, y_test)

xgb_model = XGBRegressor(**best_params)
mae, rmse, r2 = evaluate_model(xgb_model, x_train, y_train, x_test, y_test)

model_evaluation["XGBoost"] = {
    "best_params": best_params,
    "r2": r2,
    "rmse": rmse,
    "mae": mae
}

#### Dummy Models

In [ ]:
y_pred_mean = np.full_like(y_test, fill_value=y_train.mean(), dtype=float)

model_evaluation["Baseline (Mean)"] = {
    "best_params": None,
    "mae": mean_absolute_error(y_test, y_pred_mean),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred_mean)),
    "r2": r2_score(y_test, y_pred_mean)
}

In [ ]:
y_pred_persistence = x_test['bikes_available_to_rent'].values

model_evaluation["Baseline (Persistence t = t+1)"] = {
    "best_params": None,
    "mae": mean_absolute_error(y_test, y_pred_persistence),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred_persistence)),
    "r2": r2_score(y_test, y_pred_persistence)
}

### 4.3 Evaluation

In [ ]:
station_status_eval = pd.read_csv("../eval/stations_status_eval.csv")
station_eval = pd.read_csv("../eval/stations_eval.csv")

In [ ]:
evaluation = preprocessing_status(station_status_eval, station_eval)
evaluation = add_features(evaluation)
evaluation_x = evaluation[FEATURES]

y_pred = xgb_model.predict(evaluation_x)
y_true = evaluation[TARGET]

model_evaluation["Evaluation"] = {
    "best_params": None,
    "mae": mean_absolute_error(y_true, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
    "r2": r2_score(y_true, y_pred)
}

### 4.4 Display Results

In [ ]:
df_evaluation = pd.DataFrame.from_dict(model_evaluation, orient='index')

colonnes_ordre = ['r2', 'rmse', 'mae', 'best_params']
df_evaluation = df_evaluation[colonnes_ordre]

pd.set_option('display.max_colwidth', None)    
pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

display(df_evaluation.sort_values(by='r2', ascending=False))